# Question 1 — **(a)** and **(b)**

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)


In [2]:
df = pd.read_csv(RAW / "train.csv")
print("Raw dataset:", df.shape)
display(df.head())

Raw dataset: (5847, 14)


,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


## (a) Look for the missing values in all the columns and either impute them (replace with mean,median, or mode) or drop them. Justify your action for this task. 
- `New_Price` has very high missingness, so filling it with the median would manufacture many prices. Dropping that column instead. 
- For the other numerical fields, using the median because it is less affected by extreme values. (Updated in (b))

In [3]:
missing = pd.DataFrame({"Missing": df.isna().sum(), "Percent": (df.isna().mean()*100).round(2)})
display(missing)

# Drop an index exported with the CSV and the mostly missing New_Price field.
df = df.drop(columns=["Unnamed: 0", "New_Price"])

,Missing,Percent
Unnamed: 0,0,0.00
Name,0,0.00
Location,0,0.00
Year,0,0.00
Kilometers_Driven,0,0.00
Fuel_Type,0,0.00
Transmission,0,0.00
Owner_Type,0,0.00
Mileage,2,0.03
Engine,36,0.62


## (b) Remove the units from some of the attributes and only keep the numerical values (for example remove kmpl from “Mileage”, CC from “Engine”, bhp from “Power”, and lakh from “New_price”).
- Extracting the number from `Mileage`, `Engine`, and `Power`. 

In [4]:
for col in ["Mileage", "Engine", "Power"]:
    df[col] = pd.to_numeric(df[col].astype("string").str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False), errors="coerce")

for col in ["Mileage", "Engine", "Power", "Seats"]:
    df[col] = df[col].fillna(df[col].median())

print("Missing values after cleaning:")
print(df.isna().sum())
display(df[["Mileage", "Engine", "Power", "Seats"]].head())

Missing values after cleaning:
Name                 0
Location             0
Year                 0
Kilometers_Driven    0
Fuel_Type            0
Transmission         0
Owner_Type           0
Mileage              0
Engine               0
Power                0
Seats                0
Price                0
dtype: int64


,Mileage,Engine,Power,Seats
0,19.67,1582,126.2,5.0
1,13.0,1199,88.7,5.0
2,20.77,1248,88.76,7.0
3,15.2,1968,140.8,5.0
4,23.08,1461,63.1,5.0


## Save preprocessed data

In [5]:
df.to_csv(PROCESSED / "used_cars_preprocessed.csv", index=False)